# Lab 01: RAG พื้นฐานและ Hybrid Search
1 ตุลาคม 09.00–10.00 น. (ประมาณ 50 นาที)

ใน lab นี้เราจะเอาเอกสาร PDF ของโครงการมาทำให้ "ค้นหาได้" ตั้งแต่อ่านไฟล์ ทำความสะอาดข้อความ หั่นเป็นชิ้น แปลงเป็นเวกเตอร์ เก็บลง Qdrant ไปจนถึงค้นหาและวัดผลว่าค้นเจอถูกหรือไม่

ทั้งหมดนี้คือส่วน Retrieval ของ RAG ซึ่งเป็นฐานของ agent ที่เราจะทำกันต่อหลังจากนี้ ถ้าส่วนนี้ค้นไม่เจอ ต่อให้ LLM เก่งแค่ไหนก็ตอบถูกไม่ได้

```
 1. PDF ──▶ 2. ข้อความ ──▶ 3. chunk ──▶ 4. เวกเตอร์ 2 แบบ ──▶ 5. Qdrant
                                          ├ dense  (ความหมาย)       │
                                          └ sparse (คำตรงตัว)       │
 7. วัดผล ◀── 6. ค้นหา (dense / sparse / hybrid) ◀── คำถามผู้ใช้
```

| ขั้น | รับอะไรเข้า | ได้อะไรออกมา |
|---|---|---|
| 1. อ่าน PDF | ไฟล์ .pdf | ข้อความแยกตามหน้า |
| 2. ทำความสะอาด | ข้อความดิบ | ข้อความที่อ่านถูก |
| 3. Chunk | ข้อความ 1 หน้า | ข้อความสั้นหลายชิ้น พร้อมชื่อไฟล์และเลขหน้า |
| 4. Embed | ข้อความ 1 ชิ้น | dense vector (ความหมาย) และ sparse vector (คำตรงตัว) |
| 5. เก็บ | เวกเตอร์ + ข้อความ + ข้อมูลประกอบ | 1 point ใน Qdrant |
| 6. ค้นหา | คำถาม | chunk ที่ใกล้ที่สุด k ชิ้น |
| 7. วัดผล | ชุดคำถามพร้อมคำตอบที่ถูก | คะแนน hit@k |

ตลอด notebook นี้เราจะตามดูประโยคหนึ่งจากหน้า 2 ของกำหนดการ คือ "การติดตั้งและดูแลบริการด้วย Docker" ว่าแต่ละขั้นมันถูกเปลี่ยนเป็นอะไร จนสุดท้ายถูกค้นเจอได้อย่างไร

## 0. เตรียมพร้อม
ค่าที่ต้องใช้ทั้งหมด เช่น URL ของ embedding server, Qdrant และชื่อ collection ของกลุ่ม อ่านมาจากไฟล์ `.env` เราจึงไม่ต้องแก้อะไรในโค้ด

ถ้า cell นี้ขึ้น `ModuleNotFoundError` แปลว่าเลือก kernel ผิด ให้กด Select Kernel ที่มุมขวาบนแล้วเลือก `.venv`

In [ ]:
from cdd_rag.config import settings
from cdd_rag.ingest import list_pdfs

print("collection :", settings.collection)
print("embedding  :", settings.embed_base_url, settings.embed_model)
print("qdrant     :", settings.qdrant_url)
print("PDF        :", [p.name for p in list_pdfs(settings.data_dir)])

TRACE = "Docker"                                         # คำสั้น ๆ ไว้หาในข้อความดิบ
TRACE_SENTENCE = "การติดตั้งและดูแลบริการด้วย Docker"      # ประโยคที่เราจะตามดูตลอด notebook

---
## 1. อ่าน PDF
คอมพิวเตอร์ไม่ได้ "อ่าน" PDF แบบที่เราเห็นบนจอ เราต้องดึงข้อความออกมาก่อน และข้อความที่ได้ออกมาก็ไม่จำเป็นต้องตรงกับที่ตาเราเห็นเสมอไป

ก่อนรัน ลองเดาดูว่าข้อความที่ดึงออกมาจะเหมือนในไฟล์ทุกตัวอักษรหรือเปล่า

In [ ]:
import pymupdf

pdf_path = settings.data_dir / "กำหนดการโครงการ.pdf"
pages = [page.get_text() for page in pymupdf.open(pdf_path)]   # 1 หน้า = 1 ข้อความ

print(f"{pdf_path.name}: {len(pages)} หน้า\n")
print(pages[0][:250])

In [ ]:
# ตามรอย: หาบรรทัดที่มีคำว่า Docker ในข้อความดิบ
for page_no, text in enumerate(pages, start=1):
    for line in text.splitlines():
        if TRACE in line:
            print(f"หน้า {page_no}: {line}")

จะเห็นว่า "กำหนดการ" กลายเป็น "ก ำหนดกำร" และประโยคของเรากลายเป็น "กำรติดตั้งและดูแลบริกำรด้วย Docker"

ถ้าผู้ใช้พิมพ์ถามว่า "การติดตั้ง" ระบบจะหาไม่เจอ เพราะในข้อมูลไม่มีคำนี้อยู่เลย

## 2. ทำความสะอาดข้อความภาษาไทย
ปัญหานี้เกิดจากฟอนต์ในเอกสารราชการบางไฟล์ที่ map ตัวอักษรไว้ผิด ทำให้สระ า ทุกตัวถูกดึงออกมาเป็น ำ ส่วน ำ ตัวจริงจะมีช่องว่างเกินมาข้างหน้า ลองนับดู

In [ ]:
raw = "".join(pages)
print("จำนวน ำ :", raw.count("ำ"))
print("จำนวน า :", raw.count("า"), "  (เอกสารภาษาไทยทั้งไฟล์ไม่มีสระ า เลย แปลว่าผิดปกติ)")

ฟังก์ชันแก้อยู่ใน `src/cdd_rag/ingest.py` รายละเอียด regex ไม่ใช่เนื้อหาหลักของวันนี้ เรียกใช้ได้เลย

หลักการคือตรวจก่อนว่าไฟล์นี้มีอาการหรือไม่ แล้วค่อยแก้ ไฟล์ที่ปกติอยู่แล้วจะไม่ถูกแตะ

In [ ]:
from cdd_rag.ingest import normalize_thai, has_broken_sara_aa

print("ตรวจพบอาการ:", has_broken_sara_aa(raw))
print("ไฟล์ปกติไม่ถูกแก้:", normalize_thai("การดำเนินการ นนทบุรี"))

clean_pages = [normalize_thai(p) for p in pages]

# ตามรอย
for line in clean_pages[1].splitlines():
    if TRACE in line:
        print("\nหลังแก้:", line)

บทเรียนของขั้นนี้คือ ให้เปิดดูข้อความที่ดึงได้ด้วยตาทุกครั้งก่อนทำขั้นต่อไป ถ้าข้อมูลผิดตั้งแต่ต้น ไม่มีโมเดลไหนช่วยได้

ลองคิดดู: ถ้าข้ามขั้นนี้ไป แล้วผู้ใช้ถามว่า "กำหนดการอบรม" ระบบจะเจอหน้านี้ไหม

### 2.1 ข้อจำกัด: ตารางใน PDF
ตอนนี้เราดึงทุกอย่างออกมาเป็นข้อความธรรมดา รวมถึงตารางด้วย ตารางในกำหนดการมีแค่สองคอลัมน์ คือเวลากับหัวข้อ พอดึงออกมาแต่ละแถวเรียงลงมาเป็นบรรทัด ความหมายจึงยังต่อกันอยู่และค้นได้ดี

แต่ถ้าเป็นตารางที่มีหลายคอลัมน์ เช่น ตารางงบประมาณที่มีโครงการ ปี งบ และหน่วยงาน ตัวเลขจะหลุดออกจากหัวคอลัมน์ ระบบไม่รู้ว่า "1,500,000" เป็นงบของโครงการไหน ปีไหน และตารางยาว ๆ อาจถูกหั่นกลางคันจนครึ่งหลังไม่มีหัวคอลัมน์ติดไปด้วย

pymupdf มีฟังก์ชันสำหรับดึงตารางโดยเฉพาะ ลองดูว่ากับเอกสารภาษาไทยได้ผลอย่างไร

In [ ]:
import contextlib, io

page = pymupdf.open(pdf_path)[1]
with contextlib.redirect_stdout(io.StringIO()):      # ซ่อนข้อความแนะนำของ pymupdf
    tables = page.find_tables().tables

print("หน้า 2 เจอตาราง", len(tables), "ตาราง")
for row in tables[0].extract()[:4]:
    print("  ", [normalize_thai(cell or "")[:30].replace("\n", " / ") for cell in row])

ฟังก์ชันนี้หาตารางเจอถูกต้อง แต่ข้อความภาษาไทยในช่องกลับพังอีกแบบ สระและวรรณยุกต์หลุดตำแหน่ง เช่น "หัวขอ้" หรือ "ทบทวนควำมร้ / ู" ซึ่งตัวแก้ที่เราเขียนไว้ในขั้นที่ 2 แก้ไม่ได้ และช่องที่ผสานกัน (merged cell) ก็ถูกแยกเป็นหลายช่อง เครื่องมือสำเร็จรูปจึงยังใช้กับ PDF ราชการภาษาไทยทันทีไม่ได้ ต้องปรับจูนเพิ่มเติม

ถ้าเอกสารที่ต้องใช้มีตารางสำคัญ มีทางเลือกดังนี้

- ใช้ไฟล์ต้นฉบับแทน PDF ถ้ามีไฟล์ Word หรือ Excel ต้นฉบับ ให้อ่านจากไฟล์นั้นโดยตรง (เช่น python-docx หรือ pandas) จะได้โครงสร้างตารางครบและข้อความภาษาไทยถูกต้อง เพราะไม่ต้องเดาจากตำแหน่งตัวอักษรบนหน้ากระดาษ วิธีนี้แม่นที่สุด
- ใช้ OCR หรือโมเดลที่อ่าน layout ของหน้า โดยแปลงหน้า PDF เป็นภาพแล้วให้โมเดลอ่านทั้งข้อความและโครงสร้างตาราง วิธีนี้ยังจำเป็นสำหรับ PDF ที่เป็นภาพสแกน ซึ่งดึงข้อความตรง ๆ ไม่ได้เลย แต่ต้องเลือกเครื่องมือที่รองรับภาษาไทย ช้ากว่า และต้องตรวจความถูกต้องทุกครั้ง
- ปรับวิธีเก็บ เมื่อได้ตารางที่ถูกต้องแล้ว ควรเก็บทั้งตารางไว้ใน chunk เดียวในรูปแบบตาราง Markdown หรือแปลงแต่ละแถวเป็นประโยคที่มีหัวคอลัมน์ติดไปด้วย เช่น "โครงการ: กองทุนชุมชน ปี: 2566 งบ: 1,500,000 บาท" และเพิ่ม field บอกว่า chunk นี้เป็นข้อความหรือตาราง
- ถ้าคำถามต้องคำนวณตัวเลข เช่น "งบรวมทุกปีเท่าไหร่" retrieval ช่วยไม่ได้ เพราะทำได้แค่ค้นหา ควรเก็บตารางตัวเลขไว้ในฐานข้อมูลแบบตาราง แล้วให้ agent มีเครื่องมืออีกตัวไว้ query

ใน lab นี้เราใช้ข้อความธรรมดาไปก่อน เพราะตารางในกำหนดการเรียบง่าย แต่ก่อนนำไปใช้กับเอกสารจริงของหน่วยงาน ควรสำรวจก่อนว่ามีตารางมากแค่ไหนและสำคัญแค่ไหน

---
## 3. Chunk: หั่นเอกสารเป็นชิ้น
ทำไมต้องหั่น มีเหตุผลสองข้อ ข้อแรก เอกสาร 100 หน้าส่งให้ LLM ทีเดียวไม่ได้เพราะ context window มีจำกัด (ที่เรียนไปเมื่อวาน) ข้อสอง ถ้าเก็บทั้งหน้าเป็นก้อนเดียว หน้าที่มีหลายเรื่องปนกันจะมีความหมายเบลอ ทำให้ค้นได้ไม่แม่น

เราจึงหั่นเป็นชิ้นขนาดพอดี เรียกว่า chunk และทุกชิ้นต้องจำได้ว่ามาจากไฟล์ไหน หน้าไหน เพื่อใช้อ้างอิงตอนตอบ (Lab 03 ใช้เลขหน้านี้เปิด PDF ไปหน้าที่ถูกต้อง)

ค่าที่ต้องตั้งมีสองค่า
- `chunk_size` ความยาวสูงสุดของแต่ละชิ้น นับเป็นตัวอักษร
- `overlap` ส่วนที่ให้ซ้อนกับชิ้นก่อนหน้า กันไม่ให้ประโยคขาดกลางทาง

In [ ]:
from cdd_rag.ingest import split_text

page2 = clean_pages[1]
for size in (300, 600, 1500):
    print(f"chunk_size={size:>4}  หน้า 2 ได้ {len(split_text(page2, chunk_size=size, overlap=size // 5))} ชิ้น")

ในงานจริงเราใช้ `chunk_pdf()` ซึ่งทำขั้นที่ 1 ถึง 3 ให้ในครั้งเดียว และทำเพิ่มอีกสองอย่าง อย่างแรกคือตัดหัวกระดาษที่ซ้ำกันทุกหน้าออก ไม่อย่างนั้น chunk ต้นหน้าจะมีแต่หัวกระดาษ อย่างที่สองคือแปะข้อมูลประกอบ (metadata) ให้ทุกชิ้น เช่น ชื่อไฟล์ หน้า ประเภทเอกสาร และหัวข้อ (`section`) โดยข้อมูลระดับเอกสารอ่านมาจากไฟล์ `data/raw/_manifest.csv`

In [ ]:
from cdd_rag.ingest import chunk_pdf

chunks = []
for pdf in list_pdfs(settings.data_dir):              # ทุก PDF ในโฟลเดอร์ data/raw
    chunks += chunk_pdf(pdf, chunk_size=600, overlap=120)
print("ทั้งหมด", len(chunks), "chunks")

# ตามรอย: ประโยคของเราอยู่ใน chunk ไหน
found = [c for c in chunks if TRACE_SENTENCE in c["text"]]
print("ประโยคนี้อยู่ใน", [c["chunk_id"] for c in found],
      "(อยู่หลายชิ้นเพราะ overlap)" if len(found) > 1 else "")

trace_chunk = found[-1]
print()
for k in ("chunk_id", "file_name", "page", "chunk_index", "doc_type", "section"):
    print(f"{k:12}: {trace_chunk[k]}")
print("-" * 40)
print(trace_chunk["text"][:250])

สังเกตว่าในเนื้อหา (`text`) ของ chunk นี้ไม่มีคำว่า "วันที่ 3 (30 กันยายน 2569)" เลย เพราะหัวข้อวันอยู่ก่อนหน้าไปไกล ถ้ามีคนถามว่า "Docker เรียนวันไหน" ข้อความในชิ้นนี้อย่างเดียวตอบไม่ได้

ที่แก้ไว้คือ `chunk_pdf()` จะจำหัวข้อล่าสุดไว้ใน field `section` ตอนแปลงเป็นเวกเตอร์เราส่ง "หัวข้อ + เนื้อหา" เข้าไป เวกเตอร์จึงรู้ว่าเนื้อหานี้อยู่ใต้หัวข้อไหน ส่วนตอนแสดงผลหรืออ้างอิงก็ยังใช้ `text` ตามเดิม

สรุปขั้นนี้: chunk เล็กไปจะขาดบริบท ใหญ่ไปจะปนหลายเรื่อง และทุกชิ้นต้องมี `file_name` กับ `page` เสมอ

ลองคิดดู: ถ้าไม่เก็บ `page` ไว้ เราจะเสียความสามารถอะไรไป

In [ ]:
from cdd_rag.schema import embedding_text

print("context_header:", trace_chunk["context_header"])
print("\nข้อความที่จะส่งไปแปลงเป็นเวกเตอร์:\n" + embedding_text(trace_chunk)[:200])

---
## 4. Embedding: แปลงความหมายเป็นตัวเลข
สมมติผู้ใช้ถามว่า "เบรกกี่โมง" แต่ในเอกสารเขียนว่า "อาหารว่าง" ถ้าค้นแบบจับคำตรงตัวจะหาไม่เจอ

embedding แก้ปัญหานี้ด้วยการแปลงข้อความเป็นพิกัดบน "แผนที่ความหมาย" ข้อความที่ความหมายใกล้กันจะอยู่ใกล้กันบนแผนที่ แม้จะใช้คำต่างกันหรือเป็นคนละภาษา ผลลัพธ์ที่ได้คือตัวเลขชุดหนึ่ง เรียกว่า embedding หรือเวกเตอร์

โมเดลที่เราใช้คือ bge-m3 ซึ่งรองรับทั้งภาษาไทยและอังกฤษ รันเป็น API อยู่บน server และเรียกใช้แบบเดียวกับ OpenAI API

In [ ]:
from openai import OpenAI

embed_client = OpenAI(base_url=settings.embed_base_url, api_key=settings.embed_api_key)

resp = embed_client.embeddings.create(model=settings.embed_model, input=[trace_chunk["text"]])
vector = resp.data[0].embedding

# ตามรอย: chunk ของเรากลายเป็นตัวเลข
print("จำนวนตัวเลข:", len(vector))
print("5 ตัวแรก   :", [round(x, 4) for x in vector[:5]], "...")

ตัวเลขแต่ละตัวอ่านความหมายตรง ๆ ไม่ได้ สิ่งที่มีความหมายคือระยะห่างระหว่างเวกเตอร์ ซึ่งวัดด้วย cosine similarity ค่ายิ่งใกล้ 1 แปลว่าความหมายยิ่งใกล้กัน

ก่อนรัน ลองเดาดูว่า "อบรมการใช้งาน Docker" จะใกล้กับประโยคไหนที่สุด และไกลจากประโยคไหนที่สุด

In [ ]:
import math

def embed(texts):
    resp = embed_client.embeddings.create(model=settings.embed_model, input=texts)
    return [d.embedding for d in resp.data]

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

query = "อบรมการใช้งาน Docker"
candidates = [
    "Service Deployment & Operations with Docker",   # ภาษาอังกฤษ ความหมายเดียวกัน
    "การติดตั้งและดูแลบริการด้วยคอนเทนเนอร์",          # ภาษาไทย ไม่มีคำว่า Docker
    "พักรับประทานอาหารกลางวัน",                        # ไม่เกี่ยวกัน
]
q_vec, *c_vecs = embed([query] + candidates)
for text, vec in zip(candidates, c_vecs):
    print(f"{cosine(q_vec, vec):.3f}  {text}")

embedding ทำให้เราวัด "ความใกล้ของความหมาย" เป็นตัวเลขได้ จึงค้นเจอแม้คำไม่ตรงกันหรือต่างภาษา

ลองคิดดู: ตอนค้นหา คำถามของผู้ใช้ต้องแปลงเป็นเวกเตอร์ด้วยโมเดลเดียวกับตอนเก็บเอกสารหรือไม่ เพราะอะไร

### 4.2 Sparse vector (BM25): ค้นแบบจับคำ
dense vector เก่งเรื่องความหมาย แต่ไม่ค่อยแม่นกับตัวเลข รหัส หรือชื่อเฉพาะ เช่น "ห้อง 303" เลขระเบียบ หรือตัวย่อของหน่วยงาน เพราะโมเดลมองสิ่งเหล่านี้เป็น "ความหมายคล้าย ๆ กัน" ไม่ได้มองว่าต้องตรงตัว

วิธีที่ search engine ใช้กันมานานคือ BM25 ซึ่งนับคำที่ตรงกันระหว่างคำถามกับเอกสาร มีสามเรื่องที่ต้องรู้
- ภาษาไทยไม่เว้นวรรคระหว่างคำ จึงต้องตัดคำก่อน เราใช้ pythainlp
- แต่ละคำคือหนึ่งมิติ chunk หนึ่งมีคำอยู่ไม่กี่สิบคำ มิติที่เหลือเป็นศูนย์ทั้งหมดจึงไม่ต้องเก็บ เวกเตอร์แบบนี้เรียกว่า sparse (เบาบาง) ต่างจาก dense (หนาแน่น) ที่ทุกตัวมีค่า
- คำที่หายาก เช่น "303" หรือ "Docker" มีน้ำหนักมากกว่าคำทั่วไปอย่าง "การ" หรือ "วัน" ค่าน้ำหนักนี้เรียกว่า IDF และ Qdrant คำนวณให้เอง

In [ ]:
from cdd_rag.sparse import tokenize, sparse_doc, token_id

# ตามรอย: ตัดคำ chunk ของเรา
tokens = tokenize(embedding_text(trace_chunk))
print("ตัดคำ:", tokens[:15], "...")

sv = sparse_doc(embedding_text(trace_chunk))
print(f"\nsparse vector มีค่าแค่ {len(sv.indices)} มิติ (dense มี 1,024 มิติ และทุกตัวมีค่า)")
for t in dict.fromkeys(tokens[:8]):       # 8 คำแรก (ไม่ซ้ำ)
    print(f"   {t:12}  มิติ {token_id(t):>10}  น้ำหนัก {sv.values[sv.indices.index(token_id(t))]:.2f}")

| | Dense (bge-m3) | Sparse (BM25) |
|---|---|---|
| หน้าตา | ตัวเลข 1,024 ตัว ทุกตัวมีค่า | มีค่าเฉพาะคำที่ปรากฏ ไม่กี่สิบตัว |
| จับอะไร | ความหมาย | คำตรงตัว |
| เก่งเรื่อง | คำพ้อง ถามอ้อม ทับศัพท์ ข้ามภาษา | ตัวเลข รหัส ชื่อเฉพาะ |
| อ่อนเรื่อง | ตัวเลขหรือรหัสที่ต้องตรงเป๊ะ | ใช้คำต่างจากเอกสาร พิมพ์ผิด ตัดคำไม่ตรงกัน |

ลองคิดดู: ถ้าผู้ใช้พิมพ์ "อูบุนตู" แต่เอกสารเขียนว่า "Ubuntu" แบบ sparse จะหาเจอไหม แล้วแบบ dense ล่ะ

---
## 5. เก็บลง Vector Database (Qdrant)
ส่วนนี้ทบทวนสั้น ๆ รายละเอียดอยู่ในช่วงทฤษฎีแล้ว

การค้นหาคือการเอาเวกเตอร์ของคำถามไปเทียบกับเวกเตอร์ของทุก chunk ตอนนี้เรามีแค่ไม่กี่ชิ้นจะเก็บใน list ของ Python ก็ได้ แต่เอกสารทั้งกรมฯ อาจมีเป็นล้านชิ้น และมีคนค้นพร้อมกันหลายคน เราจึงต้องการที่เก็บที่ค้นได้เร็ว เก็บข้อมูลประกอบได้ กรองได้ และเก็บไว้ถาวร นั่นคือ vector database

ถ้าเทียบกับฐานข้อมูลแบบตาราง (SQL) ที่คุ้นเคยกัน

| Qdrant | เทียบกับ SQL | ในงานของเรา |
|---|---|---|
| collection | table | `cdd_docs_g01` (หนึ่งกลุ่มใช้หนึ่ง collection) |
| point | row | chunk หนึ่งชิ้น |
| id | primary key | สร้างจาก `chunk_id` ใส่ซ้ำก็ไม่เกิดข้อมูลซ้ำ |
| vector | ไม่มีใน SQL | `dense` และ `sparse` (point หนึ่งมีเวกเตอร์ได้หลายแบบ) |
| payload | columns (เก็บเป็น JSON) | `file_name`, `page`, `text` ฯลฯ |

index ใน Qdrant มีสองแบบ แบบแรกคือ vector index (HNSW) ช่วยให้หาเวกเตอร์ที่ใกล้ที่สุดได้เร็ว Qdrant สร้างให้เองเมื่อข้อมูลมากพอ แบบที่สองคือ payload index ช่วยให้กรองตามเงื่อนไขได้เร็ว เช่น เฉพาะไฟล์นี้ แบบนี้เราต้องสั่งสร้างเองตาม field ที่จะใช้กรอง

### 5.1 ออกแบบ field: แต่ละ chunk ต้องเก็บอะไร และเก็บไปเพื่ออะไร
payload ของ Qdrant จะใส่อะไรก็ได้ ไม่มี template บังคับ แต่ field ที่เก็บไว้เฉย ๆ ไม่ได้ทำให้ค้นแม่นขึ้น ก่อนเก็บจึงต้องถามว่า field นี้จะถูกใช้ตอนไหน ซึ่งแบ่งได้เป็นสี่หน้าที่

| หน้าที่ | ใช้ตอน | field ของเรา | ถ้าไม่มีจะเกิดอะไร |
|---|---|---|---|
| 1) ช่วยให้ค้นเจอ | ตอนแปลงเป็นเวกเตอร์ | `text`, `section` | chunk เรื่อง Docker ไม่รู้ว่าอยู่ใต้ "วันที่ 3" |
| 2) ช่วยกรอง | ก่อนค้น (ต้องมี payload index) | `doc_type`, `issued_date`, `file_name`, `page` | ค้นปนทุกเอกสาร เช่น ได้ระเบียบฉบับที่ยกเลิกไปแล้ว |
| 3) ช่วยอ้างอิง | หลังค้นเจอ | `file_name`, `file_path`, `page`, `text` | ตอบได้แต่บอกไม่ได้ว่ามาจากไหน ผู้ใช้ก็ไม่เชื่อ |
| 4) ช่วยดูแลระบบ | ตอน index ซ้ำหรือเติมบริบท | `chunk_id`, `doc_id`, `chunk_index`, `pipeline` | ใส่ซ้ำแล้วข้อมูลซ้ำ ไม่รู้ว่า index ด้วยค่าอะไร ดึง chunk ข้างเคียงไม่ได้ |

ค่าของ field มาจากหลายที่ `page` กับ `text` ได้จากตัว PDF ตอนอ่านไฟล์ `section` ได้จากโครงสร้างเอกสาร โดยจับบรรทัดที่เป็นหัวข้อ เช่น "วันที่ 3 (...)" "หมวด 2" หรือ "ข้อ 12" ส่วน `doc_type` กับ `issued_date` ให้คนกรอกไว้ในไฟล์ `_manifest.csv` เราไม่ให้ LLM เดา เพราะช้าและได้ค่าไม่เหมือนเดิมทุกครั้ง และสุดท้าย `doc_id` กับ `chunk_id` ระบบคำนวณเอง ใช้ตรวจว่าไฟล์เปลี่ยนหรือยัง

ก่อนจะเพิ่ม field ใหม่ ให้ถามตัวเองสามข้อ
1. field นี้ทำหน้าที่ไหนในสี่ข้อข้างบน ถ้าตอบไม่ได้ก็ไม่ต้องเก็บ
2. ได้ค่ามาจากไหน ต้องได้ครบทุกเอกสาร และได้ค่าเดิมทุกครั้งที่ index
3. วัดแล้วดีขึ้นจริงไหม เช่น การใส่ชื่อเอกสารลงในข้อความที่ embed ฟังดูน่าจะดี แต่วัดแล้ว hit@1 กลับตก (ดูเฉลยในขั้นที่ 7)

สำหรับเอกสารราชการ field ที่อาจต้องมีเพิ่มคือ `status` (ใช้อยู่ ยกเลิก หรือถูกแทนที่) หน่วยงานเจ้าของเรื่อง และคำย่อคู่กับคำเต็ม เช่น "พช." กับ "กรมการพัฒนาชุมชน"

In [ ]:
# ตามรอย: payload ของ chunk เรา แยกตามหน้าที่
roles = {
    "1) ช่วยให้ค้นเจอ":  ["section", "text"],
    "2) ช่วยกรอง":      ["doc_type", "issued_date", "file_name", "page"],
    "3) ช่วยอ้างอิง":    ["file_path", "page"],
    "4) ช่วยดูแลระบบ":   ["chunk_id", "doc_id", "chunk_index", "pipeline", "ingested_at"],
}
for role, fields in roles.items():
    print(role)
    for f in fields:
        value = str(trace_chunk[f]).replace("\n", " ")
        print(f"   {f:12} = {value[:70]}{'...' if len(value) > 70 else ''}")

### 5.2 สร้าง collection
ตอนสร้าง collection ต้องบอกว่ามีเวกเตอร์ชื่ออะไรบ้าง ขนาดเท่าไร (ต้องตรงกับโมเดล) และวัดระยะแบบไหน ค่าเหล่านี้แก้ภายหลังไม่ได้ ถ้าจะเปลี่ยนต้องสร้าง collection ใหม่ ส่วน payload เพิ่มหรือแก้ทีหลังได้ตลอด

In [ ]:
import warnings
from qdrant_client import QdrantClient, models

# ส่ง API key ผ่าน http ได้ เพราะ Qdrant เปิดเฉพาะในเครื่อง server และเข้าผ่าน SSH จึงปิดคำเตือนนี้
warnings.filterwarnings("ignore", message="Api key is used with an insecure connection")
qdrant = QdrantClient(url=settings.qdrant_url, api_key=settings.qdrant_api_key)
COLLECTION = settings.collection

if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)          # เริ่มใหม่ทุกครั้งที่รัน lab

qdrant.create_collection(
    COLLECTION,
    vectors_config={
        "dense": models.VectorParams(size=1024, distance=models.Distance.COSINE),   # 1024 คือขนาดของ bge-m3
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(modifier=models.Modifier.IDF),   # BM25: ให้ Qdrant คำนวณ IDF
    },
)
print("สร้าง", COLLECTION, "แล้ว")

ลองทำให้พังดูสักครั้ง ถ้าสร้าง collection ขนาด 768 (ซึ่งเป็นขนาดของโมเดลอื่น) แล้วเอาเวกเตอร์จาก bge-m3 ใส่ลงไป จะเกิดอะไรขึ้น

In [ ]:
qdrant.create_collection("demo_768", vectors_config={"dense": models.VectorParams(size=768, distance=models.Distance.COSINE)})
try:
    qdrant.upsert("demo_768", points=[models.PointStruct(id=1, vector={"dense": vector}, payload={})])
except Exception as e:
    print("เกิด error:", str(e)[:150])
finally:
    qdrant.delete_collection("demo_768")

### 5.3 ใส่ข้อมูล (upsert)
chunk หนึ่งชิ้นจะกลายเป็น point หนึ่งจุด ซึ่งประกอบด้วย id เวกเตอร์ทั้งสองแบบ และ payload

In [ ]:
import uuid

texts = [embedding_text(c) for c in chunks]          # หัวข้อ + เนื้อหา
vectors = embed(texts)                               # ข้อมูลน้อย ส่งทีเดียวได้

points = [
    models.PointStruct(
        id=str(uuid.uuid5(uuid.NAMESPACE_URL, c["chunk_id"])),   # id คงที่ สร้างจาก chunk_id
        vector={"dense": v, "sparse": sparse_doc(t)},             # ความหมาย และคำ
        payload=dict(c),                                          # chunk_id, file_name, page, section, text ฯลฯ
    )
    for c, v, t in zip(chunks, vectors, texts)
]
qdrant.upsert(COLLECTION, points=points)
print("ใน Qdrant มี", qdrant.count(COLLECTION).count, "points")

In [ ]:
# ตามรอย: ตอนนี้ประโยคของเราเป็น point หนึ่งจุดใน Qdrant แล้ว
trace_id = str(uuid.uuid5(uuid.NAMESPACE_URL, trace_chunk["chunk_id"]))
p = qdrant.retrieve(COLLECTION, ids=[trace_id], with_vectors=True)[0]
print("id      :", p.id)
print("dense   :", [round(x, 4) for x in p.vector["dense"][:5]], "...")
print("sparse  :", len(p.vector["sparse"].indices), "คำ")
for k in ("chunk_id", "file_name", "file_path", "page", "doc_type", "issued_date", "section"):
    print(f"payload.{k:11}: {p.payload[k]}")

ถ้าอยากเห็นข้อมูลในหน้าจอ forward port Qdrant ของเรา (`QDRANT_PORT` ใน `.env` เช่น 16307) ในแท็บ PORTS ของ VS Code แล้วเปิด `http://localhost:16307/dashboard` (ใส่ `QDRANT_API_KEY` ตอนถาม) แล้วเลือก collection ของกลุ่มตัวเอง จะเห็น point ทั้งหมดพร้อม payload

### 5.4 สร้าง payload index สำหรับ field ที่จะใช้กรอง
field ที่ agent หรือผู้ใช้น่าจะใช้กรอง ได้แก่ ไฟล์ ประเภทเอกสาร วันที่ออก และหน้า ส่วน `doc_id` กับ `chunk_index` ใช้ตอนดึง chunk ข้างเคียงมาเติมบริบท

In [ ]:
indexes = {
    "file_name":   models.PayloadSchemaType.KEYWORD,    # ค้นเฉพาะไฟล์
    "doc_type":    models.PayloadSchemaType.KEYWORD,    # ค้นเฉพาะ "ระเบียบ"
    "issued_date": models.PayloadSchemaType.DATETIME,   # ฉบับหลังปี ...
    "page":        models.PayloadSchemaType.INTEGER,
    "doc_id":      models.PayloadSchemaType.KEYWORD,    # ลบหรือแทนที่เอกสาร
    "chunk_index": models.PayloadSchemaType.INTEGER,    # ดึง chunk ก่อนและหลัง
}
for field, schema in indexes.items():
    qdrant.create_payload_index(COLLECTION, field_name=field, field_schema=schema)
print(sorted(qdrant.get_collection(COLLECTION).payload_schema.keys()))

สรุปขั้นนี้: ต้องออกแบบก่อนสร้าง ทั้งขนาดเวกเตอร์ วิธีวัดระยะ payload ที่ต้องเก็บ และ field ที่ต้องกรอง รายละเอียดการออกแบบทั้งหมดอยู่ใน `docs/index-schema.md`

ลองคิดดู: ถ้าวันหน้าเปลี่ยน embedding model ต้องทำอะไรกับ collection เดิม

---
## 6. ค้นหา (Retrieve)

### 6.1 Dense search (ค้นตามความหมาย)
ขั้นตอนคือแปลงคำถามเป็นเวกเตอร์ด้วยโมเดลเดียวกับตอนเก็บ แล้วให้ Qdrant หา point ที่ใกล้ที่สุด `top_k` จุด ผลที่ได้กลับมาคือ chunk พร้อม payload

In [ ]:
def my_search(question, top_k=3, file_name=None):
    q_vec = embed([question])[0]
    flt = None
    if file_name:
        flt = models.Filter(must=[models.FieldCondition(key="file_name", match=models.MatchValue(value=file_name))])
    res = qdrant.query_points(COLLECTION, query=q_vec, using="dense", query_filter=flt, limit=top_k, with_payload=True)
    return res.points

# ตามรอย: ถามแบบที่คนพิมพ์จริง แล้วดูว่าประโยคของเรากลับมาไหม
for rank, p in enumerate(my_search("วันไหนเรียนเรื่อง Docker"), start=1):
    mark = "   <-- chunk ที่เราตามรอย" if p.payload["chunk_id"] == trace_chunk["chunk_id"] else ""
    print(f"#{rank}  {p.score:.3f}  {p.payload['file_name']} หน้า {p.payload['page']}{mark}")

ครบเส้นทางแล้ว ประโยคจากหน้า 2 ที่ตอนแรกดึงออกมาแล้วพัง ถูกแก้ หั่นเป็น chunk แปลงเป็นเวกเตอร์ เก็บเป็น point ใน Qdrant และตอนนี้ถูกค้นเจอเป็นอันดับต้น ๆ

ลองถามด้วยคำที่ไม่มีอยู่ในเอกสารดูบ้าง

In [ ]:
for question in ["พักเบรกดื่มกาแฟกี่โมง", "ใครสอนเรื่องความปลอดภัยของ AI"]:
    print("คำถาม:", question)
    for p in my_search(question, top_k=2):
        print(f"   {p.score:.3f}  หน้า {p.payload['page']} | {p.payload['text'][:70].replace(chr(10), ' ')}")

หน้าที่ได้ถูกต้อง แต่ข้อความที่แสดงเป็นแค่ช่วงต้นของ chunk คำตอบจริง ("อาหารว่าง 10.30 น.") อยู่ท้าย chunk ลอง `print(p.payload["text"])` ดูทั้งชิ้น

อีกเรื่องที่น่าสังเกตคือคะแนนอันดับ 1 กับ 2 ห่างกันนิดเดียว แปลว่าระบบไม่ค่อยมั่นใจ ต่างจากคำถามเรื่อง Docker ที่อันดับ 1 นำห่าง

### 6.2 Sparse search และ Hybrid search
dense กับ sparse เก่งคนละแบบ ถ้าเลือกใช้แบบเดียวก็จะพลาดคำถามอีกแบบ วิธีแก้คือค้นทั้งสองแบบแล้วรวมผล เรียกว่า hybrid search

ปัญหาคือคะแนนของสองแบบอยู่คนละสเกล dense อยู่ระหว่าง 0 ถึง 1 ส่วน sparse ไม่มีเพดาน เอามาบวกกันตรง ๆ ไม่ได้ เราจึงใช้แค่อันดับ ด้วยวิธีที่เรียกว่า Reciprocal Rank Fusion (RRF)

`คะแนนรวม = 1/(60 + อันดับใน dense) + 1/(60 + อันดับใน sparse)`

chunk ที่ติดอันดับดีทั้งสองแบบจะได้คะแนนรวมสูงสุด ใน Qdrant ทำได้ในคำสั่งเดียว โดยใช้ `prefetch` ค้นทั้งสองแบบ แล้วใช้ `FusionQuery(RRF)` รวมผล

In [ ]:
from cdd_rag.sparse import sparse_query

def my_sparse_search(question, top_k=3):
    res = qdrant.query_points(COLLECTION, query=sparse_query(question), using="sparse", limit=top_k, with_payload=True)
    return res.points

def my_hybrid_search(question, top_k=3):
    res = qdrant.query_points(
        COLLECTION,
        prefetch=[
            models.Prefetch(query=embed([question])[0], using="dense", limit=20),    # ผู้สมัครจาก dense
            models.Prefetch(query=sparse_query(question), using="sparse", limit=20), # ผู้สมัครจาก sparse
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),                          # รวมด้วยอันดับ
        limit=top_k,
        with_payload=True,
    )
    return res.points

ก่อนรัน ลองเดาว่าคำถามสามข้อนี้ แต่ละแบบจะหาเจอหรือไม่
1. "ห้อง 303" เป็นตัวเลขที่ต้องตรงตัว
2. "อูบุนตู" แต่เอกสารเขียนว่า "Ubuntu"
3. "สอนเรื่องคอนเทนเนอร์วันไหน" แต่เอกสารเขียนว่า "Docker"

In [ ]:
tests = [("ห้อง 303", "303"), ("อูบุนตู", "Ubuntu"), ("สอนเรื่องคอนเทนเนอร์วันไหน", "Docker")]
searchers = {"dense": my_search, "sparse": my_sparse_search, "hybrid": my_hybrid_search}

for question, must_contain in tests:
    print(f"คำถาม: {question}   (chunk ที่ถูกต้องต้องมีคำว่า '{must_contain}')")
    for name, fn in searchers.items():
        pts = fn(question)
        if not pts:
            print(f"   {name:7} ✘ ไม่เจอเลย (ไม่มีคำไหนตรง)")
            continue
        top = pts[0].payload
        ok = must_contain.lower() in top["text"].lower()
        print(f"   {name:7} {'✔' if ok else '✘'} อันดับ 1 คือหน้า {top['page']} | {top['text'][:45].replace(chr(10), ' ')}")
    print()

ผลที่ได้ตอนเตรียม lab คือ "ห้อง 303" แบบ dense พลาด เพราะมองตัวเลขเป็นแค่ความหมายคล้าย ๆ ส่วน sparse เจอเพราะคำตรงตัว กลับกัน "อูบุนตู" กับ "คอนเทนเนอร์" แบบ sparse ไม่เจอเลย เพราะไม่มีคำนี้ในเอกสาร แต่ dense เข้าใจความหมาย ส่วน hybrid ถูกทั้งสามข้อ

อย่างไรก็ตาม hybrid ไม่ได้ชนะทุกครั้ง ใน Lab 02 เราจะวัดบน benchmark 1,190 คำถาม แล้วจะเห็นว่าต้องวัดบนข้อมูลของเราเองเสมอ

ลองคิดดู: เอกสารของกรมฯ มีเลขระเบียบ ชื่อโครงการ และตัวย่อเยอะ ควรเลือกใช้แบบไหน

### 6.3 ใช้ field ช่วยค้น: กรองและเติมบริบท
field ที่ทำหน้าที่ "ช่วยกรอง" และ "ช่วยดูแลระบบ" ในหัวข้อ 5.1 ถูกใช้ตรงนี้

In [ ]:
from cdd_rag.store import search, expand

q = "สอนเรื่อง Docker วันไหน"
print("ไม่กรอง               :", [(h["file_name"], h["page"]) for h in search(q, top_k=2)])
print("กรอง doc_type=กำหนดการ :", [(h["file_name"], h["page"]) for h in search(q, top_k=2, doc_type="กำหนดการ")])
print("กรอง doc_type=ระเบียบ   :", search(q, top_k=2, doc_type="ระเบียบ"), "(ยังไม่มีเอกสารประเภทนี้ จึงไม่ได้อะไรกลับมา)")
print("กรอง ออกหลัง 2027-01-01 :", search(q, top_k=2, date_from="2027-01-01"))

In [ ]:
# เติมบริบท: chunk ที่ค้นเจออาจตัดกลางเนื้อหา จึงดึง chunk ก่อนและหลังมาด้วย (ใช้ doc_id + chunk_index)
top = search(q, top_k=1)[0]
print("ค้นเจอ:", top["chunk_id"], "|", top["section"])
for h in expand(top, window=1):
    mark = "   <-- ที่ค้นเจอ" if h["chunk_id"] == top["chunk_id"] else ""
    print(f"   {h['chunk_id']}  หน้า {h['page']}  {h['text'][:40].replace(chr(10), ' ')}{mark}")

เมื่อมีเอกสารหลายร้อยไฟล์ การกรองทำให้ไม่ต้องไปแข่งกับเอกสารที่ไม่เกี่ยวข้องเลย ในช่วงบ่าย agent จะเป็นคนเลือกเองว่าจะกรองด้วยอะไรจากคำถามของผู้ใช้

สรุปคือ field ที่ดีต้องมีหน้าที่ชัด ได้ค่ามาอย่างเชื่อถือได้ และวัดแล้วช่วยได้จริง

### 6.4 โค้ดที่เราเขียนมา ถูกเก็บไว้ที่ไหน
ฟังก์ชันในขั้นที่ 4 ถึง 6 ที่เพิ่งเขียนกัน ถูกรวมไว้ในไฟล์ `.py` เพื่อให้ agent (Lab 03), MCP server และ service ที่ขึ้น Docker เรียกใช้ตัวเดียวกัน แก้ที่เดียวแล้วทุกที่ได้ผลเหมือนกัน

```
from cdd_rag.store import search
     โฟลเดอร์   ไฟล์     ฟังก์ชัน     =   src/cdd_rag/store.py  ฟังก์ชัน search(...)
```

| ที่เราเขียนใน notebook | ของจริงใน `src/cdd_rag/` |
|---|---|
| `embed()` | `embed.py` ฟังก์ชัน `embed()` (แบ่งส่งทีละชุดให้ด้วย) |
| `create_collection` และ payload index | `store.py` ฟังก์ชัน `ensure_collection()` |
| `upsert` | `store.py` ฟังก์ชัน `upsert_chunks()` และ `ingest_file()` (ใส่ซ้ำได้ ไม่ embed ซ้ำ) |
| `sparse_doc()` และ `sparse_query()` | `sparse.py` |
| `my_search()`, `my_sparse_search()`, `my_hybrid_search()` | `store.py` ฟังก์ชัน `search(mode="dense" / "sparse" / "hybrid")` |

ลองกด Cmd หรือ Ctrl ค้างไว้แล้วคลิกที่คำว่า `search` ใน cell ข้างล่าง VS Code จะเปิดไฟล์ไปที่ฟังก์ชันนั้นให้ทันที

In [ ]:
from cdd_rag.store import search

for mode in ("dense", "hybrid"):
    print(mode, [(h["file_name"], h["page"]) for h in search("วันไหนเรียนเรื่อง Docker", top_k=3, mode=mode)])
# ผลเหมือน my_search() และ my_hybrid_search() ด้านบน เพราะเป็นโค้ดเดียวกัน
# ถ้าไม่ระบุ mode ค่าเริ่มต้นของ search() คือ "hybrid"

---
## 7. วัดผล: ค้นเจอหน้าที่ถูกหรือไม่
การลองถามสองสามคำถามแล้วรู้สึกว่า "น่าจะใช้ได้" ยังไม่พอ เราต้องวัดออกมาเป็นตัวเลข จะได้เทียบได้ว่าปรับแล้วดีขึ้นหรือแย่ลง

เครื่องมือที่ใช้คือ golden set ซึ่งเป็นชุดคำถามแบบที่ผู้ใช้จริงจะถาม จับคู่กับคำตอบที่ถูก (ไฟล์และหน้า) ที่คนตรวจแล้ว ส่วนคะแนน hit@k คือสัดส่วนของคำถามที่หน้าที่ถูกต้องอยู่ใน k อันดับแรก

ข้อสำคัญคือคำถามต้องไม่ลอกประโยคมาจากเอกสาร ไม่อย่างนั้นค้นเจอแน่นอนและวัดอะไรไม่ได้

In [ ]:
golden = [
    # (คำถามแบบที่คนพิมพ์จริง,                ไฟล์ที่ถูก,             หน้าที่ถูก)
    ("การอบรมเรื่อง Docker จัดวันไหน",          "กำหนดการโครงการ.pdf", 2),
    ("Design Thinking สอนช่วงเวลาไหน",           "กำหนดการโครงการ.pdf", 1),
    ("หัวข้อ Human-in-the-loop อยู่วันไหน",        "กำหนดการโครงการ.pdf", 2),
    ("ใครสอนเรื่องความปลอดภัยของระบบ AI",        "กำหนดการโครงการ.pdf", 3),
    ("ปิดโครงการกี่โมง",                          "กำหนดการโครงการ.pdf", 3),
    ("พักเบรกดื่มกาแฟเวลาเท่าไหร่",                 "กำหนดการโครงการ.pdf", 3),
    ("การเทียบเคียงแนวทางปฏิบัติที่เป็นเลิศ",        "กำหนดการโครงการ.pdf", 1),
    ("model serving ด้วย vLLM",                   "กำหนดการโครงการ.pdf", 2),
    # คำถามยาก: ตัวเลขตรงตัว ทับศัพท์ และใช้คำต่างจากเอกสาร
    ("ห้อง 303",                                  "กำหนดการโครงการ.pdf", 1),
    ("อูบุนตู",                                    "กำหนดการโครงการ.pdf", 2),
    ("สอนเรื่องคอนเทนเนอร์วันไหน",                  "กำหนดการโครงการ.pdf", 2),
]

def is_hit(question, file_name, page, k, mode):
    return (file_name, page) in [(h["file_name"], h["page"]) for h in search(question, top_k=k, mode=mode)]

modes = ("dense", "sparse", "hybrid")

# รายข้อ: อันดับ 1 ถูกหรือไม่
print(f"{'คำถาม':38}" + "".join(f"{m:>8}" for m in modes))
for q, f, p in golden:
    print(f"{q:38}" + "".join(f"{'✔' if is_hit(q, f, p, 1, m) else '✘':>8}" for m in modes))

# สรุป
print()
for k in (1, 3):
    print(f"hit@{k}".ljust(38) + "".join(f"{sum(is_hit(q, f, p, k, m) for q, f, p in golden) / len(golden):>8.2f}" for m in modes))

คะแนนที่ออกมาสูง ส่วนหนึ่งเป็นเพราะเอกสารมีแค่ 3 หน้า ถ้าเป็นงานจริง ต้องมีเอกสารหลายไฟล์ที่เนื้อหาใกล้เคียงกัน และมีคำถามยาก ๆ ด้วย เช่น ถามอ้อม ต้องใช้ข้อมูลหลายส่วนประกอบกัน หรือถามเรื่องที่ไม่มีในเอกสาร ใน Lab 02 เราจะวัดผลกันแบบเต็มรูปแบบ

ลองทำเอง: เปลี่ยน `chunk_size` ในขั้นที่ 3 เป็น 800 แล้วรันใหม่ตั้งแต่ขั้นที่ 3 ดูว่าตารางเปลี่ยนไปอย่างไร แบบไหนพลาดเพิ่ม และเพราะอะไร

<details><summary>เฉลย (ผลที่วัดไว้ตอนเตรียม lab, hit@1)</summary>

| chunk_size | จำนวน chunk | dense | sparse | hybrid |
|---|---|---|---|---|
| 400 | 13 | 0.91 | 0.82 | 1.00 |
| 600 | 9 | 0.91 | 0.82 | 1.00 |
| 800 | 7 | 0.82 ("พักเบรก" พลาดเพิ่ม) | 0.82 | 1.00 |

ที่ขนาด 800 หน้า 3 ทั้งหน้ากลายเป็น chunk เดียว หมายเหตุเรื่อง "อาหารว่าง 10.30 น." ที่อยู่ท้ายหน้าจึงถูกเนื้อหาอื่นกลบ dense เลยพลาด แต่ hybrid ยังถูกทุกข้อ เพราะ sparse ช่วยจับคำว่า "อาหารว่าง" และ "เครื่องดื่ม" ได้ตรงตัว

อีกการทดลองที่ทำไว้คือ ถ้าแปะชื่อเอกสารไว้หน้าทุก chunk ก่อน embed คะแนน hit@1 ของ dense ตกจาก 0.88 เหลือ 0.75 เพราะทุก chunk มีข้อความยาวที่ซ้ำกัน จึงดูคล้ายกันมากขึ้น

ข้อสรุปคือ จะปรับอะไรก็ตาม ต้องวัดผลก่อนและหลังเสมอ สิ่งที่ฟังดูน่าจะดีขึ้นอาจทำให้แย่ลงก็ได้
</details>

---
## ส่งต่อให้ agent (Lab 03)
ข้อมูลใน Qdrant พร้อมใช้แล้ว ไม่ต้องแปลงอะไรเพิ่ม ใน Lab ถัดไป agent จะเรียก `search()` ตัวเดียวกันนี้ผ่าน tool ชื่อ `search_documents`

cell ข้างล่างแค่ให้ดูว่า agent จะเห็นอะไรเมื่อค้นเอกสาร ไม่บังคับรัน

In [ ]:
from cdd_rag.tools import run_search_documents

text, hits = run_search_documents("สอนเรื่อง Docker วันไหน")
print(text[:350])

---
## (เมื่อมี LLM แล้ว) ให้ LLM ตอบพร้อมอ้างอิง
chunk ที่ค้นเจอจะถูกส่งให้ LLM พร้อมเลขอ้างอิงในรูปแบบ `[n] ไฟล์ หน้า p` เพื่อให้ LLM ใส่เลขอ้างอิงไว้ในคำตอบ

In [ ]:
from cdd_rag.rag import build_context, answer

print(build_context(search("วันไหนเรียนเรื่อง Docker", top_k=2))[:400])

if settings.llm_model:
    res = answer("การอบรมเรื่อง Docker จัดวันที่เท่าไหร่ และใครเป็นวิทยากร")
    print("\n" + res["answer"])
else:
    print("\n(ข้าม เพราะยังไม่ได้ตั้ง LLM_MODEL ใน .env)")

---
## สรุป

| ขั้น | สิ่งที่ควรจำ |
|---|---|
| 1–2 อ่านและทำความสะอาด | เปิดดูข้อความด้วยตาก่อนเสมอ ภาษาไทยพังง่าย |
| 3 Chunk | ทุกชิ้นต้องมี `file_name` และ `page` |
| 4 Dense และ Sparse | dense จับความหมาย sparse จับคำตรงตัว และภาษาไทยต้องตัดคำก่อน |
| 5 Vector DB | ออกแบบ field ก่อนสร้าง ทุก field ต้องมีหน้าที่ชัดเจน |
| 6 ค้นหา | hybrid รวมอันดับของสองแบบด้วย RRF และให้ดู chunk ที่ได้ด้วยตาก่อนจะโทษ LLM |
| 7 วัดผล | ใช้ golden set วัดเป็นตัวเลข และเทียบทุกแบบ |

## คำศัพท์

| คำ | ความหมาย |
|---|---|
| RAG | Retrieval-Augmented Generation ค้นเอกสารที่เกี่ยวข้องก่อน แล้วให้ LLM ตอบจากเอกสารนั้น |
| Chunk | ข้อความชิ้นเล็กที่หั่นมาจากเอกสาร |
| Metadata | ข้อมูลประกอบของ chunk เช่น ชื่อไฟล์ เลขหน้า |
| Embedding / เวกเตอร์ | ตัวเลขชุดหนึ่ง (1,024 ตัวสำหรับ bge-m3) ที่แทนความหมายของข้อความ |
| Cosine similarity | ค่าความใกล้ของเวกเตอร์สองตัว ยิ่งใกล้ 1 ความหมายยิ่งใกล้กัน |
| Dense vector | เวกเตอร์ที่ทุกตัวมีค่า ใช้แทนความหมาย |
| Sparse vector / BM25 | เวกเตอร์ที่มีค่าเฉพาะคำที่ปรากฏ ใช้ค้นแบบจับคำ |
| ตัดคำ (tokenize) | แยกข้อความภาษาไทยออกเป็นคำ ๆ ก่อนทำ sparse (ใช้ pythainlp) |
| IDF | น้ำหนักของคำ คำหายากมีน้ำหนักมาก คำทั่วไปมีน้ำหนักน้อย |
| Hybrid search | ค้นทั้ง dense และ sparse แล้วรวมผล |
| RRF | Reciprocal Rank Fusion วิธีรวมผลจากหลายวิธีโดยใช้อันดับแทนคะแนนดิบ |
| Vector database | ฐานข้อมูลที่เก็บเวกเตอร์พร้อมข้อมูลประกอบ และค้นตามความใกล้ได้เร็ว |
| Collection / Point / Payload | ตาราง แถว และข้อมูลประกอบใน Qdrant |
| Field | ข้อมูลแต่ละช่องใน payload ต้องมีหน้าที่ คือช่วยค้น ช่วยกรอง ช่วยอ้างอิง หรือช่วยดูแลระบบ |
| Manifest | ไฟล์ที่คนกรอกข้อมูลระดับเอกสาร เช่น ประเภทและวันที่ออก (`data/raw/_manifest.csv`) |
| Indexing (กระบวนการ) | เส้นทางทั้งหมดตั้งแต่อ่าน หั่น แปลงเป็นเวกเตอร์ จนเก็บลงฐานข้อมูล |
| Index (โครงสร้างข้อมูล) | สิ่งที่ฐานข้อมูลสร้างไว้เพื่อให้ค้นเร็ว เช่น HNSW สำหรับเวกเตอร์ และ payload index สำหรับการกรอง |
| top_k | จำนวนผลลัพธ์ที่ขอจากการค้นหา |
| Golden set | ชุดคำถามพร้อมคำตอบที่ถูก ใช้วัดผล |
| hit@k | สัดส่วนคำถามที่คำตอบถูกอยู่ใน k อันดับแรก |

ต่อไป Lab 02 เราจะวัดผลกันแบบจริงจัง ส่วน Lab 03 จะเอา `search()` ไปเป็นเครื่องมือของ docs agent